# GaugePredict Figure Creation Notebook  
Caitlin R. R. Turner, January 2026  

This notebook reproduces two figure workflows:

1. **Training and test performance figure**: loads saved model runs for selected horizons and plots the training history and test time series comparison.
2. **SHAP geoplot figure**: builds a map grid showing SHAP-selected predictor gauge locations by horizon.

- The notebook assumes a GaugePredict-style project layout with `examples/` containing `results/` and `shapefiles/`.


## Import necessary dependencies

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import json

from GaugePredict.routines import get_project_root, resolve_under_project
from GaugePredict.plotting import (
    load_saved_runs,
    parameter_label_from_target,
    build_aligned_test_series,
    get_horizon_styles,
    build_scores_table,
    plot_training_and_timeseries,
    plot_shap_geoplot_grid,
)


## Resolve project paths

Here we define the base directories used throughout the notebook. The goal is to reliably locate the `examples/` folder (inputs and outputs) and the `examples/results/` folder (saved model runs, figures, and SHAP outputs).

### Project root

- **`project_root`**: The top-level GaugePredict project directory used as the anchor for all relative paths.  
  In a notebook, `__file__` is not defined, so the code starts from the current working directory (`Path.cwd()`) and uses `get_project_root(...)` to walk upward and identify the repository root.

If your notebook is not located inside the repository, set `project_root` manually, for example:
**`project_root = Path(r"C:/path/to/GaugePredict").resolve()`**

### Examples directory

- **`examples_dir`**: Path to the `examples/` folder under the project root. This directory is treated as the main workspace for inputs (cached datasets, shapefiles) and outputs (results folders).

### Results directory

- **`results_base_dir`**: Path to `examples/results/`. This is the base directory that holds:

  - trained run folders (for example, `examples/results/<target_site>_<run_name>/`)
  - SHAP run folders (for example, `examples/results/<target_site>_test_final/`)
  - figures saved by this notebook

We will then double check all of the paths.

In [ ]:
project_root = get_project_root(Path.cwd(), levels_up=1)
examples_dir = resolve_under_project(project_root, Path("examples"))
results_base_dir = examples_dir / "results"
print("project_root:", project_root)
print("examples_dir:", examples_dir)
print("results_base_dir:", results_base_dir)

## Training and test performance figure

Now we will load saved model outputs for a completed training run and produces a combined figure showing training behavior and test-set time series performance across forecast horizons.

### Key settings

- **`target_site`**: Gauge identifier for the site being predicted. This must match the folder naming used during training.
- **`run_name`**: Run folder suffix used during training. The notebook expects results under:  
  `examples/results/<target_site>_<run_name>/`
- **`target_variable`**: Target variable label used to build axis labels (via `parameter_label_from_target(...)`).
- **`horizons`**: Forecast horizons (in days) to include in both the plot and the score table.
- **`site_label`**: Display label used in the figure annotation (human-readable site name).
- **`save_fig`**: If `True`, writes the figure to disk in the run results directory.

In [ ]:
target_site = "01280"
run_name = "final_bcs_noroll"
target_variable = "waterlevel"
horizons = [1, 3]
site_label = "Bonnet Carré Spillway (USACE)"
save_fig = True

### Outputs and paths

- **`results_root`**: The directory containing saved run outputs for this target and run name:  
  `examples/results/<target_site>_<run_name>/`
- **`fig_path`**: Output filename for the training/test figure saved into `results_root`.

In [ ]:
results_root = results_base_dir / f"{target_site}_{run_name}"
fig_path = results_root / "fig_training_test_agu.png"

### Load saved runs

- **`load_saved_runs(results_root, horizons)`** reads the saved per-horizon outputs (predictions, metrics, training history, and any stored metadata) from disk.  
  This assumes the training notebook or script already created the expected files under `results_root`.

In [ ]:
results = load_saved_runs(results_root, horizons, verbose=True)

### Align test series across horizons

Forecast horizons produce predictions with different valid date ranges. To compare them on the same timeline:

- **`build_aligned_test_series(results, horizons)`** returns:
  - **`date_index`**: datetime index for the aligned test period
  - **`y_true`**: observed target values over that same index
  - **`pred_df`**: a DataFrame of predictions with one column per horizon, aligned to `date_index`

In [ ]:
date_index, y_true, pred_df = build_aligned_test_series(results, horizons)

### Styling by horizon

- **`get_horizon_styles(horizons)`** assigns consistent colors and line styles for each horizon so plots remain comparable across figures and runs.
- **`parameter_label_from_target(target_variable)`** converts the target variable name into a readable axis label (units and formatting depend on your plotting utilities).

In [ ]:
colors_h, linestyles_h = get_horizon_styles(horizons)
parameter_label = parameter_label_from_target(target_variable)

### Plot training history and test time series

- **`plot_training_and_timeseries(...)`** produces a combined figure that typically includes:
  - training behavior (loss curves or other tracked history, depending on what was saved)
  - test-set time series with observed vs predicted values for each horizon

The `roll_window_days` argument controls optional smoothing of plotted series to make easier to visualize. Report in results if used. (set to 1 here, meaning no effective rolling average).

If `save_fig = True`:

- The run directory is created if needed.
- The figure is saved at high resolution (600 dpi) with tight bounding boxes for publication-friendly output.

In [ ]:
fig1, ax1 = plot_training_and_timeseries(
    results=results,
    horizons=horizons,
    date_index=date_index,
    y_true=y_true,
    pred_df=pred_df,
    colors_h=colors_h,
    linestyles_h=linestyles_h,
    parameter_label=parameter_label,
    roll_window_days=1,
    site=site_label,
)

if save_fig:
    results_root.mkdir(parents=True, exist_ok=True)
    fig1.savefig(fig_path, dpi=600, bbox_inches="tight")
    print("Saved:", fig_path)

plt.show()

### Test score table

- **`build_scores_table(results, horizons)`** assembles a table of evaluation metrics for the test set for each horizon.
- The printed output provides a quick numeric summary alongside the plotted performance. Results will not be impacted by rolling window. 

In [ ]:
scores_df = build_scores_table(results, horizons)
print(f"\nTest set scores ({target_site}, {target_variable}):")
print(scores_df.round(3))

## SHAP geoplot figure

Here we will create a map based summary of SHAP site selection. It reads SHAP artifacts from a SHAP run folder under `examples/results/` and plots a grid of maps showing which predictor gauges are available and which gauges are selected for each forecast horizon.

### Key settings

- **`target_site`**: Gauge identifier used to construct the SHAP run folder name.
- **`shap_run_name`**: Folder name containing SHAP artifacts. A common convention is `"<target_site>_test_final"`.
- **`full_shap_root`**: Full path to the SHAP run folder under `examples/results/`. This directory is expected to contain the SHAP outputs used for plotting.
- **`horizons`**: Horizons (in days) to include in the map grid. Only horizons listed here are plotted.
- **`n_shap_by_h`**: Dictionary mapping horizon → number of SHAP-selected predictor sites to highlight for that horizon.  
  Horizons not plotted can remain in the dictionary so the configuration stays consistent across notebooks.
- **`states_fp`**: Shapefile path for US state boundaries used as the basemap reference layer (Optional).
- **`xlim`, `ylim`**: Longitude and latitude bounds for the plotted map extent. These should be adjusted if you want a wider view or a regional zoom.
- **`fig_path`**: Output file path for the SHAP geoplot figure.
- **`save_fig`**: If `True`, writes the figure to `fig_path`. If `False`, the figure is shown but not saved.

In [ ]:
shap_run_name = f"{target_site}_test_final"
full_shap_root = results_base_dir / shap_run_name
horizons = [1, 3]

n_shap_by_h = {
    1: 5,
    3: 9}

states_fp = examples_dir / "shapefiles" / "US_STATES" / "tl_2023_us_state.shp"

xlim = (-115.0, -76.0)
ylim = (28.5, 50.0)

fig_path = full_shap_root / "fig_shap_agu_paper.png"
save_fig = False

### Figure layout and sizing

- **`nrows`, `ncols`**: Grid layout used to arrange horizon panels.  
  The number of plotted horizons should be consistent with the grid arrangement you choose.
- **`fig_w`, `fig_h`**: Figure size in inches. These values are tuned for paper-friendly output.

In [ ]:
ncols = 2
nrows = 1
fig_w = 6.25
fig_h = 1.75

### Target marker

- **`target_lon`, `target_lat`**: Longitude and latitude used to mark the prediction site on the map.
- **`site_label`**: Label shown next to the target marker (line breaks are allowed).

In [ ]:
target_lon = -91.23
target_lat = 30.45

### Marker sizes and spacing

- **`s_all`**: Marker size for all available predictor sites.
- **`s_used`**: Marker size for the SHAP-selected subset (typically larger for emphasis).
- **`wspace`, `hspace`**: Panel spacing controls for the grid.
- **`cbar_rect`**: Colorbar position and size given as a rectangle tuple `(left, bottom, width, height)` in figure coordinates.

### Plot call

- **`plot_shap_geoplot_grid(...)`** builds the full multi-panel figure. It reads from `full_shap_root`, plots the requested horizons, overlays state boundaries, highlights SHAP-selected sites per horizon using `n_shap_by_h`, and optionally saves the figure if `save_fig` is enabled.

In [ ]:
plot_shap_geoplot_grid(
    shap_root=full_shap_root,
    horizons=horizons,
    n_shap_by_h=n_shap_by_h,
    states_fp=states_fp,
    xlim=xlim,
    ylim=ylim,
    fig_w=fig_w,
    fig_h=fig_h,
    nrows=nrows,
    ncols=ncols,
    s_all=6.0,
    s_used=18.0,
    wspace=0.03,
    hspace=-0.0125,
    cbar_rect=(0.125, 0.07, 0.775, 0.03),
    save_path=fig_path if save_fig else None,
    target_lon=target_lon,
    target_lat=target_lat,
    target_label=site_label,
    show=True,
)

if save_fig:
    print("Saved:", fig_path)


If you have any questions or need help with implementation to your model, please do not hesitate to contact Caitlin R. R. Turner at cturn65@lsu.edu

## Acknowledgments

Research reported in this publication was supported by the US Department of Defense and Army Engineer Research and Development Center (ERDC) under Contract No. W912HZ2220005, the Gulf Research Program of the National Academies of Sciences, Engineering, and Medicine under award number SCON-10000883, and the NSF through Open Earthscape (Collaborative Research: Frameworks: OpenEarthscape, Transformative Cyberinfrastructure for Modeling and Simulation in the Earth-Surface Science Communities) award No. 2104102.